# Price Determining Function

Creating a simple function that determines price according to merit order data.

In [8]:
# Let's load in the necessary packages
import pandas as pd
import datetime
import os
import timeit
import matplotlib.pyplot as plt
from ortools.linear_solver import pywraplp
# from ortools.sat.python import cp_model
import numpy as np

#### Function from Prof. Leach
Will use here the code that Professor Leach gave.
It's a function that gets the new price for a certain hour.

In [15]:
# Loading in the data first.
# Let's load in the data here, and for now, let's make it just for four hours.

file_name = os.fsdecode("marco_data.csv")
workbook = pd.read_csv(file_name)
workbook = workbook.filter(items=["date", "he", "size","flexible","price"]) # Gets the columns that we need

workbook = workbook.query('date=="2024-04-10"'and 'he==15') # Remove this to take out the filter for the day.

# Getting the columns we need.
merit_data = workbook.filter(items=["date", "he", "size","flexible","price"]) # Gets the columns that we need

#pd.set_option('display.max_rows',50)

# Temporary peek:
temp = merit_data.copy()
#temp



,date,he,size,flexible,price
3518,2024-04-10,15,0.0395,N,0.00
3519,2024-04-10,15,0.0648,N,0.00
3520,2024-04-10,15,0.0752,N,0.00
3521,2024-04-10,15,0.1373,N,0.00
3522,2024-04-10,15,0.1919,N,0.00
...,...,...,...,...,...
3785,2024-04-10,15,16.0000,Y,999.99
3786,2024-04-10,15,25.0000,Y,999.99
3787,2024-04-10,15,42.0000,N,999.99
3788,2024-04-10,15,80.0000,Y,999.99


In [28]:
# Making everything we have above into a function:
#THIS IS YOU MAKING A 3rd filtered dataset.

bids = merit_data.query('he==15')
#set ail for testing
ail = 2000

# def pricing(bids, ail): # Bids is self-explanatory, so set of bids needed.
bids=bids.sort_values('price').reset_index(drop = True)
# bids=bids.assign(merit=bids['size'].cumsum())
bids['merit'] = bids['size'].cumsum() # Old code is above

bids = bids.assign(surplus=bids['merit']-ail)

#bids['surplus'] = bids['merit'] - ail
bids


,date,he,size,flexible,price,merit,surplus
0,2024-04-10,15,0.0395,N,0.00,0.0395,-1999.9605
1,2024-04-10,15,43.0000,Y,0.00,43.0395,-1956.9605
2,2024-04-10,15,44.0000,N,0.00,87.0395,-1912.9605
3,2024-04-10,15,45.0000,N,0.00,132.0395,-1867.9605
4,2024-04-10,15,46.0000,N,0.00,178.0395,-1821.9605
...,...,...,...,...,...,...,...
267,2024-04-10,15,2.0000,Y,999.99,12101.7169,10101.7169
268,2024-04-10,15,2.0000,Y,999.99,12103.7169,10103.7169
269,2024-04-10,15,2.0000,Y,999.99,12105.7169,10105.7169
270,2024-04-10,15,5.0000,Y,999.99,12110.7169,10110.7169


In [30]:
#pick the max negative surplus
last_full=bids.query('surplus<=0')['surplus'].idxmax()
print(last_full,bids['surplus'][last_full])
# Made this change to get the actual energy needed left
e_needed=-bids['surplus'][last_full] # Getting the last negative surplus value since that's the amound lacking
print(e_needed)
print(bids.loc[last_full-2:last_full+2])

41 -80.94679999999971
80.94679999999971
          date  he      size flexible  price      merit   surplus
39  2024-04-10  15   79.0882        N    0.0  1760.2235 -239.7765
40  2024-04-10  15   79.1963        N    0.0  1839.4198 -160.5802
41  2024-04-10  15   79.6334        N    0.0  1919.0532  -80.9468
42  2024-04-10  15  139.2640        N    0.0  2058.3172   58.3172
43  2024-04-10  15  140.0000        N    0.0  2198.3172  198.3172


In [31]:
#dont' divide this stuff like this. e_needed and last_full will only be right when you run it the first time.
while(e_needed>0):
    last_full+=1 #do this here
    if(bids['size'][last_full]<e_needed): #dispatch it
        e_needed-=bids['size'][last_full]
    elif (bids['size'][last_full]>=e_needed) and (bids['flexible'][last_full]=="Y"): #dispatch part of it 
        e_needed-=e_needed
    # Made changes in the bids below to get the price accurately
    price=bids['price'][last_full-1]
    print(e_needed,last_full-1,price)
print(bids.loc[last_full-2:last_full+2])

80.94679999999971 41 0.0
80.94679999999971 42 0.0
80.94679999999971 43 0.0
0.0 44 0.0
          date  he     size flexible  price      merit   surplus
43  2024-04-10  15  140.000        N    0.0  2198.3172  198.3172
44  2024-04-10  15  145.000        N    0.0  2343.3172  343.3172
45  2024-04-10  15  153.000        Y    0.0  2496.3172  496.3172
46  2024-04-10  15  158.000        N    0.0  2654.3172  654.3172
47  2024-04-10  15  162.996        N    0.0  2817.3132  817.3132


In [44]:
bids

,date,he,size,flexible,price,ail,merit,surplus
0,2018-08-05,15,0.0081,N,0.00,8980.7614,0.0081,-1999.9919
1,2018-08-05,15,80.0000,Y,0.00,8980.7614,80.0081,-1919.9919
2,2018-08-05,15,75.0000,N,0.00,8980.7614,155.0081,-1844.9919
3,2018-08-05,15,71.0000,Y,0.00,8980.7614,226.0081,-1773.9919
4,2018-08-05,15,64.0000,N,0.00,8980.7614,290.0081,-1709.9919
5,2018-08-05,15,60.0000,N,0.00,8980.7614,350.0081,-1649.9919
6,2018-08-05,15,50.0000,Y,0.00,8980.7614,400.0081,-1599.9919
7,2018-08-05,15,45.0000,N,0.00,8980.7614,445.0081,-1554.9919
8,2018-08-05,15,43.0000,Y,0.00,8980.7614,488.0081,-1511.9919
9,2018-08-05,15,41.0000,Y,0.00,8980.7614,529.0081,-1470.9919
